In [ ]:
from halib import *

def video_hstack(video_files, output_file):
    # Create a temporary file list for FFmpeg
    with open("video_list.txt", "w") as f:
        for video in video_files:
            f.write(f"file '{video}'\n")
    # FFmpeg command to stack videos horizontally
    ffmpeg_cmd = (
        f"ffmpeg -f concat -safe 0 -i video_list.txt "
        f'-filter_complex "[0:v][1:v][2:v]hstack=inputs={len(video_files)}[v]" '
        f'-map "[v]" -c:v libx264 -preset fast -crf 22 {output_file}'
    )

    # Execute FFmpeg command
    try:
        os.system(ffmpeg_cmd)
        print(f"Video stacked successfully: {output_file}")
    except Exception as e:
        print(f"Error during video stacking: {e}")
    finally:
        # Clean up temporary file
        if os.path.exists("video_list.txt"):
            os.remove("video_list.txt")



In [64]:
METHODS = ["prof_hgnetv2b5_notemp", "yolov5s_notemp", "yolov5l_notemp"]
INDIR = "/mnt/e/SyncData/paper2_baseline/zout/DFire_test"
VIDEODIR = "/mnt/e/SyncData/paper2_main/datasets/DFire/test"

def verify_csv_video(methods=METHODS, indir=INDIR):
    rs_dict = {}
    for method in methods:
        method_dir = os.path.join(indir, method)
        pprint(method_dir)
        video_files = fs.filter_files_by_extension(VIDEODIR, ".mp4", recursive=False)
        gt_files = fs.filter_files_by_extension(VIDEODIR, ".csv", recursive=False)
        csv_files = fs.filter_files_by_extension(method_dir, ".csv", recursive=False)
        vis_files = fs.filter_files_by_extension(method_dir, ".mp4", recursive=True)
        sort_lambda = lambda x: os.path.basename(x).split("_")[0]

        gt_files = sorted(gt_files, key=sort_lambda)
        video_files = sorted(video_files, key=sort_lambda)
        csv_files = sorted(csv_files, key=sort_lambda)
        vis_files = sorted(vis_files, key=sort_lambda)
        # pprint(gt_files)
        # pprint(video_files)
        # pprint(csv_files)
        # pprint(vis_files)
        assert len(csv_files) == len(vis_files) == len(video_files) == len(gt_files), f"Mismatch in counts for method {method}"
        rs_dict[method] = {
            "csv": csv_files,
            "vis": vis_files,
            "video": video_files,
            "gt": gt_files}
    return rs_dict

rs_dict = verify_csv_video()
# pprint(rs_dict)

'/mnt/e/SyncData/paper2_baseline/zout/DFire_test/prof_hgnetv2b5_notemp'

'/mnt/e/SyncData/paper2_baseline/zout/DFire_test/yolov5s_notemp'

'/mnt/e/SyncData/paper2_baseline/zout/DFire_test/yolov5l_notemp'

In [72]:
def rs_dict_to_dataframe(rs_dict):
    # Initialize lists to store data
    data = {
        "video": [],
    }

    # Create column names for each method and file type
    for method in rs_dict.keys():
        data[f"{method}_csv"] = []
        data[f"{method}_vis"] = []
        data[f"{method}_video"] = []
        data[f"{method}_gt"] = []

    # Get video names from one of the methods (they should be the same across all)
    video_files = rs_dict[list(rs_dict.keys())[0]]["video"]
    # pprint(video_files)
    video_names = [os.path.basename(v) for v in video_files]
    # pprint(video_names)

    # Populate the data dictionary
    for i, video_name in enumerate(video_names):
        data["video"].append(video_name.split(".")[0])  # Remove file extension
        for method in rs_dict.keys():
            data[f"{method}_csv"].append(rs_dict[method]["csv"][i])
            data[f"{method}_vis"].append(rs_dict[method]["vis"][i])
            data[f"{method}_video"].append(rs_dict[method]["video"][i])
            data[f"{method}_gt"].append(rs_dict[method]["gt"][i])

    # Create DataFrame
    df = pd.DataFrame(data)
    gt_cols = [f"{method}_gt" for method in rs_dict.keys()]
    keep_first = gt_cols[0]
    for col in gt_cols[1:]:
        del df[col]
    df = df.rename(columns={keep_first: "gt"})
    # move gt to the second column
    cols = df.columns.tolist()
    cols.insert(1, cols.pop(cols.index("gt")))
    df = df[cols]

    video_columns = [f"{method}_video" for method in rs_dict.keys()]
    keep_first = video_columns[0]
    for col in video_columns[1:]:
        del df[col]

    # rename keep_first to video_path
    df = df.rename(columns={keep_first: "video_path"})
    # move video_path to the third column
    cols = df.columns.tolist()
    cols.insert(2, cols.pop(cols.index("video_path")))
    df = df[cols]

    return df

df = rs_dict_to_dataframe(rs_dict)
df

,video,gt,video_path,prof_hgnetv2b5_notemp_csv,prof_hgnetv2b5_notemp_vis,yolov5s_notemp_csv,yolov5s_notemp_vis,yolov5l_notemp_csv,yolov5l_notemp_vis
0,FP1,/mnt/e/SyncData/paper2_main/datasets/DFire/tes...,/mnt/e/SyncData/paper2_main/datasets/DFire/tes...,/mnt/e/SyncData/paper2_baseline/zout/DFire_tes...,/mnt/e/SyncData/paper2_baseline/zout/DFire_tes...,/mnt/e/SyncData/paper2_baseline/zout/DFire_tes...,/mnt/e/SyncData/paper2_baseline/zout/DFire_tes...,/mnt/e/SyncData/paper2_baseline/zout/DFire_tes...,/mnt/e/SyncData/paper2_baseline/zout/DFire_tes...
1,FP11,/mnt/e/SyncData/paper2_main/datasets/DFire/tes...,/mnt/e/SyncData/paper2_main/datasets/DFire/tes...,/mnt/e/SyncData/paper2_baseline/zout/DFire_tes...,/mnt/e/SyncData/paper2_baseline/zout/DFire_tes...,/mnt/e/SyncData/paper2_baseline/zout/DFire_tes...,/mnt/e/SyncData/paper2_baseline/zout/DFire_tes...,/mnt/e/SyncData/paper2_baseline/zout/DFire_tes...,/mnt/e/SyncData/paper2_baseline/zout/DFire_tes...
2,FP12,/mnt/e/SyncData/paper2_main/datasets/DFire/tes...,/mnt/e/SyncData/paper2_main/datasets/DFire/tes...,/mnt/e/SyncData/paper2_baseline/zout/DFire_tes...,/mnt/e/SyncData/paper2_baseline/zout/DFire_tes...,/mnt/e/SyncData/paper2_baseline/zout/DFire_tes...,/mnt/e/SyncData/paper2_baseline/zout/DFire_tes...,/mnt/e/SyncData/paper2_baseline/zout/DFire_tes...,/mnt/e/SyncData/paper2_baseline/zout/DFire_tes...
3,FP13,/mnt/e/SyncData/paper2_main/datasets/DFire/tes...,/mnt/e/SyncData/paper2_main/datasets/DFire/tes...,/mnt/e/SyncData/paper2_baseline/zout/DFire_tes...,/mnt/e/SyncData/paper2_baseline/zout/DFire_tes...,/mnt/e/SyncData/paper2_baseline/zout/DFire_tes...,/mnt/e/SyncData/paper2_baseline/zout/DFire_tes...,/mnt/e/SyncData/paper2_baseline/zout/DFire_tes...,/mnt/e/SyncData/paper2_baseline/zout/DFire_tes...
4,FP14,/mnt/e/SyncData/paper2_main/datasets/DFire/tes...,/mnt/e/SyncData/paper2_main/datasets/DFire/tes...,/mnt/e/SyncData/paper2_baseline/zout/DFire_tes...,/mnt/e/SyncData/paper2_baseline/zout/DFire_tes...,/mnt/e/SyncData/paper2_baseline/zout/DFire_tes...,/mnt/e/SyncData/paper2_baseline/zout/DFire_tes...,/mnt/e/SyncData/paper2_baseline/zout/DFire_tes...,/mnt/e/SyncData/paper2_baseline/zout/DFire_tes...
...,...,...,...,...,...,...,...,...,...
65,VP5,/mnt/e/SyncData/paper2_main/datasets/DFire/tes...,/mnt/e/SyncData/paper2_main/datasets/DFire/tes...,/mnt/e/SyncData/paper2_baseline/zout/DFire_tes...,/mnt/e/SyncData/paper2_baseline/zout/DFire_tes...,/mnt/e/SyncData/paper2_baseline/zout/DFire_tes...,/mnt/e/SyncData/paper2_baseline/zout/DFire_tes...,/mnt/e/SyncData/paper2_baseline/zout/DFire_tes...,/mnt/e/SyncData/paper2_baseline/zout/DFire_tes...
66,VP6,/mnt/e/SyncData/paper2_main/datasets/DFire/tes...,/mnt/e/SyncData/paper2_main/datasets/DFire/tes...,/mnt/e/SyncData/paper2_baseline/zout/DFire_tes...,/mnt/e/SyncData/paper2_baseline/zout/DFire_tes...,/mnt/e/SyncData/paper2_baseline/zout/DFire_tes...,/mnt/e/SyncData/paper2_baseline/zout/DFire_tes...,/mnt/e/SyncData/paper2_baseline/zout/DFire_tes...,/mnt/e/SyncData/paper2_baseline/zout/DFire_tes...
67,VP7,/mnt/e/SyncData/paper2_main/datasets/DFire/tes...,/mnt/e/SyncData/paper2_main/datasets/DFire/tes...,/mnt/e/SyncData/paper2_baseline/zout/DFire_tes...,/mnt/e/SyncData/paper2_baseline/zout/DFire_tes...,/mnt/e/SyncData/paper2_baseline/zout/DFire_tes...,/mnt/e/SyncData/paper2_baseline/zout/DFire_tes...,/mnt/e/SyncData/paper2_baseline/zout/DFire_tes...,/mnt/e/SyncData/paper2_baseline/zout/DFire_tes...
68,VP8,/mnt/e/SyncData/paper2_main/datasets/DFire/tes...,/mnt/e/SyncData/paper2_main/datasets/DFire/tes...,/mnt/e/SyncData/paper2_baseline/zout/DFire_tes...,/mnt/e/SyncData/paper2_baseline/zout/DFire_tes...,/mnt/e/SyncData/paper2_baseline/zout/DFire_tes...,/mnt/e/SyncData/paper2_baseline/zout/DFire_tes...,/mnt/e/SyncData/paper2_baseline/zout/DFire_tes...,/mnt/e/SyncData/paper2_baseline/zout/DFire_tes...


In [77]:
pprint(df.columns.tolist())
csvfile.show(df.head(2))
dfmk = csvfile.DFCreator()
POS = "O_Fire_smoke"
NEG = "X_None"

def calc_perf(
    video,
    video_gt_lb,
    total_frames,
    mt_name,
    method_pred_csv,
):
    if "prof" in mt_name:
        df_pred = pd.read_csv(
            method_pred_csv,
            sep=";",
            encoding="utf-8",
            dtype={"pred_label": str, "elapsed_time": float},
            keep_default_na=False,
        )
    else:
        df_pred = pd.read_csv(method_pred_csv, sep=";", encoding="utf-8")
    if len(df_pred) > total_frames:
        total_frames = len(df_pred)

    pred_lb = None
    correct = False
    num_wrong = 0
    if "prof" in mt_name:
        if video_gt_lb == POS:
            pred_lb_pos = df_pred[df_pred["pred_label"] != "None"]
            if len(pred_lb_pos) > 0:
                pred_lb = POS
                correct = True
                num_wrong = len(df_pred[df_pred["pred_label"] == "None"])
            else:
                pred_lb = NEG
                correct = False
                num_wrong = len(df_pred)
        elif video_gt_lb == NEG:
            pred_as_pos = df_pred[df_pred["pred_label"] != "None"]
            if len(pred_as_pos) > 0:
                pred_lb = POS
                correct = False
                num_wrong = len(pred_as_pos)
            else:
                pred_lb = NEG
                correct = True
                num_wrong = 0
        else:
            raise ValueError(f"Unknown gt label {video_gt_lb} for video {video}")
    else: # yolo cases
        df_pred.drop_duplicates(subset=["frame_id"], keep="first", inplace=True)
        if video_gt_lb == POS:
            if len(df_pred) > 0:
                pred_lb = POS
                correct = True
                num_wrong = total_frames - len(df_pred)
            else:
                pred_lb = NEG
                correct = False
                num_wrong = total_frames
        elif video_gt_lb == NEG:
            if len(df_pred) > 0:
                pred_lb = POS
                correct = False
                num_wrong = len(df_pred)
            else:
                pred_lb = NEG
                correct = True
                num_wrong = 0
        else:
            raise ValueError(f"Unknown gt label {video_gt_lb} for video {video}")

    return pred_lb, correct, num_wrong


def proc(df):
    cols = ["video", "gt", "total_frames"]
    method_cols = ["pred", "correct", "num_wrong_frames"]
    for cmethod in METHODS:
        cols += [f"{cmethod}_{mc}" for mc in method_cols]
    dfmk.create_table("perf", cols)
    rows = []
    for idx, row in df.iterrows():
        video = row["video"]
        video_gt_lb = POS if "VP" in video else NEG
        gt_csv = row["gt"]
        df = pd.read_csv(gt_csv, sep=";", encoding="utf-8")
        total_frames = len(df)
        row_data = [video, video_gt_lb, total_frames]
        for mt_name in METHODS:
            mt_csv = row[f"{mt_name}_csv"]
            pred_lb, correct, num_wrong = calc_perf(
                video,
                video_gt_lb,
                total_frames,
                mt_name,
                mt_csv,
            )
            row_data += [pred_lb, correct, num_wrong]
        rows.append(row_data)
    dfmk.insert_rows("perf", rows)
    dfmk.fill_table_from_row_pool("perf")
    final_df = dfmk["perf"]
    return final_df


final_df = proc(df)
final_df
for cmethod in METHODS:
    col = f"{cmethod}_pred"
    del final_df[col]
final_df

[
│   'video',
│   'gt',
│   'video_path',
│   'prof_hgnetv2b5_notemp_csv',
│   'prof_hgnetv2b5_notemp_vis',
│   'yolov5s_notemp_csv',
│   'yolov5s_notemp_vis',
│   'yolov5l_notemp_csv',
│   'yolov5l_notemp_vis'
]

Loading ITables v2.4.4 from the internet... (need help?)


,video,gt,total_frames,prof_hgnetv2b5_notemp_correct,prof_hgnetv2b5_notemp_num_wrong_frames,yolov5s_notemp_correct,yolov5s_notemp_num_wrong_frames,yolov5l_notemp_correct,yolov5l_notemp_num_wrong_frames
0,FP1,X_None,30,True,0,True,0,True,0
1,FP11,X_None,60,True,0,False,1,False,2
2,FP12,X_None,60,False,14,True,0,True,0
3,FP13,X_None,60,True,0,False,13,False,7
4,FP14,X_None,3,True,0,True,0,True,0
...,...,...,...,...,...,...,...,...,...
65,VP5,O_Fire_smoke,60,False,60,True,48,True,41
66,VP6,O_Fire_smoke,60,True,45,True,11,True,9
67,VP7,O_Fire_smoke,60,False,60,True,7,True,8
68,VP8,O_Fire_smoke,60,True,47,True,24,True,27


In [80]:
final_df.to_csv("./zout/comparison_results.csv", index=False, encoding="utf-8", sep=";")

correct_cls = []
for cmethod in METHODS:
    col = f"{cmethod}_correct"
    correct_cls.append(col)

# finalout the case that all methods are correct
all_correct = final_df[correct_cls].all(axis=1)

# fillter out the case that at least one method is wrong
any_wrong = ~final_df[correct_cls].all(axis=1)

all_correct_df = final_df[all_correct]
any_wrong_df = final_df[any_wrong]

all_correct_df.to_csv("./zout/all_methods_correct.csv", index=False, encoding="utf-8", sep=";")
any_wrong_df.to_csv("./zout/method_wrong.csv", index=False, encoding="utf-8", sep=";")
